# 03 · Curvas, quartis e orçamento

Os notebooks reutilizam `lorem_study`; Run All é offline e não coleta nem retreina.
Modo padrão: agregados **reais retidos de 18/10/2026**. Reanálise completa exige caminho
explícito e validação de hashes. O modo `synthetic` é somente verificação técnica.
Veja [execução e recuperação](README.md). Outputs locais ficam em `study/work/`.


In [ ]:
from pathlib import Path
import os, sys
# Resolve from repository, study or notebooks directory; no machine-specific path.
start = Path.cwd().resolve()
study = next((p if (p / "lorem_study").is_dir() else p / "study"
              for p in [start, *start.parents]
              if (p / "lorem_study").is_dir() or (p / "study/lorem_study").is_dir()), None)
if study is None:
    raise RuntimeError("Abra o notebook a partir do repositório Lorem.")
sys.path.insert(0, str(study))
# Run All never contacts HTTP services; the pipeline remains callable via external CLI.
import urllib.request
def no_network(*args, **kwargs):
    raise RuntimeError("Notebook offline: execute coleta explicitamente pela CLI em novo diretório.")
urllib.request.urlopen = no_network
from lorem_study.notebook_session import Session
from lorem_study import notebook_views as views
session = Session.from_environment(study)
views.notice("Fonte: **" + session.source['data_kind'] + "** · plano `" + session.plan_sha + "`.")


## analysis-time-bands-v2
T1=P25, T2=P50, T3=P75 com eventos e censura; **T4=h é orçamento operacional, nunca P100**.
A região central é P25–P75. Percentis apenas dos resolvidos não substituem esses quantis KM.

| Faixa observada de AC | Leitura descritiva |
|---|---|
| t < P25 | Muito rápido |
| P25 ≤ t < P50 | Mais rápido que a mediana |
| P50 ≤ t < P75 | Metade mais lenta da região central |
| P75 ≤ t ≤ h, se P75 ≤ h | Lento dentro do orçamento |
| t > h | AC depois do orçamento |

Censura antes de h deixa o desfecho em h desconhecido. Não resolver no prazo não significa
incapacidade permanente. AC **exatamente em h** é elegível; AC depois e censura são separados.
Quantil não alcançado fica null com motivo; nunca virar 120min. Empates deixam faixas vazias.
Quartil conhecido acima de h é preservado e marca orçamento incompatível, sem ajustar limites.


In [ ]:
X, Y, H = 1200, 1600, 120  # edite e execute novamente; no modo agregado só consulta retida
SECOND_X, SECOND_Y = 1600, 1600
views.quantiles(session, X, Y, H, SECOND_X, SECOND_Y)

## Caso real retido: por que não há P75 até 120min?
F(120)=0,622099<0,75 no bin histórico de delta. A mediana retida é 91,4667min.
Três probabilidades e uma mediana não identificam P25 ou a curva completa; os pontos
retidos são exibidos sem uma linha interpolada. A ausência não permite inferir incapacidade.

O contrato antigo `time-curve-v1` usa p50/p75/p90, continua desabilitado e não é modificado
por esses limiares analíticos. Não há pontuação, mudança de saldo ou decisão pedagógica aqui.